In [ ]:
# =========================
# BASIC
# =========================
import pandas as pd
import numpy as np
import pickle
import json

# =========================
# VISUALIZATION
# =========================
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# SKLEARN
# =========================
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

# =========================
# TENSORFLOW
# =========================
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    SpatialDropout1D,
    GlobalMaxPooling1D
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    Callback
)

print("TensorFlow Version:", tf.__version__)

In [ ]:
sheet_url = "https://docs.google.com/spreadsheets/d/1O_1gKmMrNAN4G2H9AJeOA0m9Yanw3L6_xrhAjNpJtZY/export?format=csv"

df = pd.read_csv(sheet_url)

print("Total Raw Data:", len(df))

df.head()

In [ ]:
# Select columns
df = df[
    [
        'product_name',
        'message_clean',
        'rating',
        'is_anonymous',
        'total_likes'
    ]
].copy()

# Drop missing
df.dropna(inplace=True)

# Remove super short reviews
df['word_count'] = df['message_clean'].apply(
    lambda x: len(str(x).split())
)

df = df[df['word_count'] >= 2]

print("Total Clean Data:", len(df))

In [ ]:
def label_sentiment(rating):
    return 1 if rating >= 4 else 0

df['label'] = df['rating'].apply(label_sentiment)

print(df['label'].value_counts())

In [ ]:
# =========================
# DUPLICATE REVIEW
# =========================
df['is_duplicate'] = df.duplicated(
    subset=['message_clean']
).astype(int)

# =========================
# GENERIC WORDS
# =========================
generic_words = [
    'bagus',
    'mantap',
    'baik',
    'cocok',
    'ok',
    'sesuai',
    'banget',
    'keren',
    'top',
    'mantul',
    'recommended'
]

def generic_score(text):
    words = str(text).split()

    return sum(
        1 for w in words if w in generic_words
    ) / (len(words) + 1)

df['generic_score'] = df['message_clean'].apply(
    generic_score
)

# =========================
# REPETITION SCORE
# =========================
def repetition_score(text):
    words = str(text).split()

    return 1 - (
        len(set(words)) / (len(words) + 1)
    )

df['repetition_score'] = df['message_clean'].apply(
    repetition_score
)

# =========================
# LOW CREDIBILITY USER
# =========================
df['low_credibility_user'] = (
    (df['is_anonymous'] == True) &
    (df['total_likes'] == 0)
).astype(int)

# =========================
# SUSPICIOUS SCORE
# =========================
df['suspicious_score'] = (
    ((1 / (df['word_count'] + 1)) * 0.30) +
    (df['generic_score'] * 0.25) +
    (df['repetition_score'] * 0.20) +
    (df['is_duplicate'] * 0.15) +
    (df['low_credibility_user'] * 0.10)
).clip(0, 1)

# =========================
# RICHNESS SCORE
# =========================
df['richness_score'] = (
    (
        df['word_count'] /
        df['word_count'].max()
    ) * 0.5
    +
    (
        1 - df['generic_score']
    ) * 0.5
)

# =========================
# TRUST SCORE
# =========================
df['sentiment_score'] = df['rating'] / 5

df['trust_score'] = (
    (df['sentiment_score'] * 0.5)
    +
    ((1 - df['suspicious_score']) * 0.3)
    +
    (df['richness_score'] * 0.2)
).clip(0, 1)

df.head()

In [ ]:
X = df['message_clean'].astype(str)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train Size:", len(X_train))
print("Test Size:", len(X_test))

In [ ]:
VOCAB_SIZE = 12000
MAX_LEN = 60

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print(X_train_pad.shape)

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(
    enumerate(class_weights)
)

print(class_weight_dict)

In [ ]:
class TrainingMonitorCallback(Callback):

    def on_epoch_end(self, epoch, logs=None):

        print(
            f"""
Epoch {epoch+1}

Train Accuracy : {logs['accuracy']:.4f}
Validation Accuracy : {logs['val_accuracy']:.4f}

Train Loss : {logs['loss']:.4f}
Validation Loss : {logs['val_loss']:.4f}
"""
        )

In [ ]:
input_layer = Input(shape=(MAX_LEN,))

x = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=64
)(input_layer)

x = SpatialDropout1D(0.3)(x)

x = Bidirectional(
    LSTM(
        32,
        return_sequences=True
    )
)(x)

x = GlobalMaxPooling1D()(x)

x = Dropout(0.4)(x)

x = Dense(
    32,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

output_layer = Dense(
    1,
    activation='sigmoid'
)(x)

model = Model(
    inputs=input_layer,
    outputs=output_layer
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import TensorBoard

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_aris_model.keras",
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# TensorBoard
tensorboard = TensorBoard(
    log_dir='logs',
    histogram_freq=1
)

custom_callback = TrainingMonitorCallback()

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,

    validation_data=(
        X_test_pad,
        y_test
    ),

    epochs=15,
    batch_size=32,

    class_weight=class_weight_dict,

    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint,
        tensorboard,
        custom_callback
        ]
)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend([
    'Train',
    'Validation'
])

plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
y_probs = model.predict(X_test_pad)

# Threshold tuning
threshold = 0.45

y_pred = (
    y_probs > threshold
).astype(int)

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
roc_auc = roc_auc_score(
    y_test,
    y_probs
)

print("ROC AUC Score:", roc_auc)

In [ ]:
model.save("aris_review_model.keras")

with open("aris_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "VOCAB_SIZE": VOCAB_SIZE,
    "MAX_LEN": MAX_LEN,
    "THRESHOLD": threshold
}

with open("model_config.json", "w") as f:
    json.dump(config, f)

print("✅ Model Saved Successfully")

In [ ]:
def predict_review(text):

    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding='post',
        truncating='post'
    )

    prediction = model.predict(
        padded,
        verbose=0
    )[0][0]

    sentiment = (
        "positive"
        if prediction >= threshold
        else "negative"
    )

    confidence = float(prediction)

    if confidence >= 0.80:
        confidence_label = "High"

    elif confidence >= 0.60:
        confidence_label = "Medium"

    else:
        confidence_label = "Low"

    return {
        "text": text,
        "sentiment": sentiment,
        "confidence": round(confidence, 4),
        "confidence_label": confidence_label
    }

In [ ]:
def analyze_product(df_product):

    results = []

    for _, row in df_product.iterrows():

        prediction = predict_review(
            row['message_clean']
        )

        results.append({

            "review":
                row['message_clean'],

            "sentiment":
                prediction['sentiment'],

            "confidence":
                prediction['confidence'],

            "suspicious_score":
                round(
                    row['suspicious_score'],
                    4
                ),

            "trust_score":
                round(
                    row['trust_score'],
                    4
                )
        })

    result_df = pd.DataFrame(results)

    summary = {

        "total_reviews":
            len(result_df),

        "positive_reviews":
            int(
                (
                    result_df['sentiment']
                    == 'positive'
                ).sum()
            ),

        "negative_reviews":
            int(
                (
                    result_df['sentiment']
                    == 'negative'
                ).sum()
            ),

        "average_confidence":
            round(
                result_df['confidence'].mean(),
                4
            ),

        "average_suspicious_score":
            round(
                result_df['suspicious_score'].mean(),
                4
            ),

        "average_trust_score":
            round(
                result_df['trust_score'].mean(),
                4
            )
    }

    return summary

In [ ]:
predict_review(
    "barang bagus banget pengiriman cepat"
)

In [ ]:
sample_product = df[
    df['product_name'].str.contains(
        "Redjelly",
        case=False
    )
]

analyze_product(sample_product.head(20))